# Generalization Study: Unseen-Cell, Temporal Extrapolation, & External Zero-Shot Transfer
## TE-Q-Transformer Research Framework

**Scientific Research Question:**  
*"Does the observed performance of TE-Q-Transformer hold on strictly unseen cells and temporal horizons, and how does a model trained on NASA cylindrical cells generalize zero-shot to prismatic CALCE cells of a different chemistry?"*

---

### Crucial Scientific Invariants:
1. **Unseen-Cell Generalization:** Cells `B0018` (132 cycles, 24°C) and `B0032` (39 cycles, 4°C) were held out entirely from training, validation, scaling fit, and hyperparameter selection.
2. **Temporal Extrapolation:** Cell `B0053` (44°C) had its first 70% (cycles 0–36) placed in the training pool and final 30% (cycles 37–52) evaluated as temporal forward forecasting.
3. **Terminology Boundary:** NASA Ames evaluations are strictly **unseen-cell** and **temporal extrapolation**, NOT "unseen-temperature" generalization, because training cells already span 4°C, 24°C, and 44°C.
4. **External CALCE Transfer:** Evaluated **strictly zero-shot** on CALCE CS2 prismatic cells (`CS2_35`, `CS2_36`, `CS2_37`, `CS2_38`) with frozen weights and frozen NASA scaler. No CALCE training, fine-tuning, or scaler refitting was performed.


In [ ]:
# ==============================================================================
# 0. CONFIGURABLE REPOSITORY ROOT PATH & ENVIRONMENT SETUP
# ==============================================================================
import os
import sys
from pathlib import Path

# Manual override: Set to Path("your/path") if needed; otherwise auto-discovered.
MANUAL_REPO_ROOT = None
REPO_NAME = "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation"

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/kaggle/working") / REPO_NAME,
    Path("/kaggle/working"),
    Path("/content") / REPO_NAME,
    Path("/content"),
]

if MANUAL_REPO_ROOT and Path(MANUAL_REPO_ROOT).exists():
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    REPO_ROOT = next(
        (c.resolve() for c in CANDIDATES if (c / "models" / "proposed" / "te_q_transformer.py").exists() or (c / "datasets" / "NASA" / "processed").exists()),
        Path.cwd().resolve()
    )

print(f"[Setup] REPO_ROOT resolved to: {REPO_ROOT}")
DATA_ROOT = REPO_ROOT / "datasets"
MODEL_ROOT = REPO_ROOT / "models"
RESULT_ROOT = REPO_ROOT / "results"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Install PennyLane if running in a fresh cloud runtime (Kaggle/Colab)
try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    import pennylane as qml

import random
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Hardware device: {DEVICE}")


## 1. Unseen-Cell & Temporal Extrapolation Across All Models


In [ ]:
gen_model_csv = RESULT_ROOT / "tables" / "generalization_model_metrics.csv"
gen_cell_csv = RESULT_ROOT / "tables" / "generalization_cell_metrics.csv"

if gen_model_csv.exists() and gen_cell_csv.exists():
    df_gen_m = pd.read_csv(gen_model_csv)
    df_gen_c = pd.read_csv(gen_cell_csv)
    print("[Overall Generalization Benchmark Metrics (All 11 Models)]:")
    display(df_gen_m.sort_values(by="overall_rmse"))
    
    print("\n[Per-Condition Breakdown (Unseen Cells vs Temporal Extrapolation)]:")
    display(df_gen_c.head(15))
else:
    print("Generalization metrics tables not found in results/tables/.")


## 2. NASA → CALCE Zero-Shot External Transfer
Prismatic LiCoO₂ cells from CALCE evaluated without retraining:
- **Macro RMSE:** **0.3302**
- **Pooled Combined RMSE:** **0.3325**
- **Pooled Combined MAE:** **0.2676**
- **Pooled Combined $R^2$:** **-1.789**

*Interpretation:* The negative $R^2$ score quantitatively demonstrates that models trained on NASA cylindrical cells cannot accurately track the degradation trajectory of CALCE prismatic cells without domain adaptation or chemistry-specific calibration. We report this negative result openly as a scientific limitation.


In [ ]:
calce_csv = RESULT_ROOT / "tables" / "calce_zero_shot_metrics.csv"

if calce_csv.exists():
    df_calce = pd.read_csv(calce_csv)
    print("[CALCE Zero-Shot Transfer Metrics]")
    display(df_calce)
    
    # Bar plot of CALCE errors
    fig, ax = plt.subplots(figsize=(8, 4))
    cell_rows = df_calce[df_calce['cell'].str.startswith('CS2_')].copy()
    ax.bar(cell_rows['cell'], cell_rows['RMSE'], color="#457B9D", label="Zero-Shot RMSE")
    ax.set_ylabel("RMSE")
    ax.set_title("Zero-Shot Cross-Dataset Transfer Errors on CALCE CS2 Cells")
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend()
    plt.tight_layout()
    plt.show()


## 3. Zero-Shot Trajectory Tracking on CALCE
Visualizing predicted vs actual capacity degradation profiles on CALCE cells.


In [ ]:
calce_fig = RESULT_ROOT / "figures" / "soh_trajectory_E01_baseline_reproduction_CS2_35.png"
if calce_fig.exists():
    from IPython.display import Image, display as ipy_display
    print("Zero-Shot SOH Trajectory on CS2_35:")
    ipy_display(Image(filename=str(calce_fig)))
else:
    print("Trajectory plots available in results/figures/.")
